# Sales Prediction - Modeling

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sqlalchemy import create_engine

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

from sklearn.linear_model import LogisticRegression, LinearRegression

from sklearn.metrics import (
    mean_absolute_error, 
    mean_squared_error, 
    r2_score,
    accuracy_score, 
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    ConfusionMatrixDisplay
)

from sklearn.preprocessing import StandardScaler

from xgboost import XGBRegressor

## 2. Dataset Overview

In [ ]:
server = "."
database = "SalesPredictionDB"

connection_string = (
    f"mssql+pyodbc://{server}/{database}"
    "?trusted_connection=yes"
    "&driver=ODBC+Driver+17+for+SQL+Server"
)

In [ ]:
engine = create_engine(connection_string)

In [ ]:
query = """
SELECT
    CustomerID,
    CutOffDate,
    TotalRevenue,
    NumberOfOrders,
    TotalQuantity,
    AverageOrderValue,
    AverageQuantityPerOrder,
    Recency,
    CustomerLifetimeDays,
    FutureRevenue
FROM VW_CustomerSalesFeatures
"""

df_ml = pd.read_sql(query, engine)

df_ml.head()

In [ ]:
df_ml.info()

In [ ]:
df_ml["CutOffDate"] = pd.to_datetime(df_ml["CutOffDate"])
df_ml["CutOffDate"].dtype

In [ ]:
df_ml.describe()

In [ ]:
numeric_cols = [
    "TotalRevenue",
    "TotalQuantity",
    "NumberOfOrders",
    "AverageOrderValue",
    "AverageQuantityPerOrder",
    "Recency",
    "CustomerLifetimeDays",
    "FutureRevenue"
]

corr = df_ml[numeric_cols].corr()

plt.figure(figsize=(10, 7))
plt.imshow(corr)
plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right")
plt.yticks(range(len(corr.columns)), corr.columns)
plt.colorbar()
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

## 3. Classification
### - Logistic Regression

In [ ]:
df_ml["FuturePurchase"] = (df_ml["FutureRevenue"] > 0).astype(int)
df_ml.head()

In [ ]:
features = [
    "TotalRevenue",
    "NumberOfOrders",
    "TotalQuantity",
    "AverageOrderValue",
    "AverageQuantityPerOrder",
    "Recency",
    "CustomerLifetimeDays"
]

x_cls = df_ml[features]
y_cls = df_ml["FuturePurchase"]

### -Train/Test split
- We used a time-based split to prevent data leakage. Data before "2011-10-01" was used for training, while data from "2011-10-01" onward was reserved for testing. We avoided a random "train_test_split" because future observations should not be used to predict past observations.

In [ ]:
msk = df_ml["CutOffDate"] < '2011-10-01'
x_train_cls = x_cls[msk]
y_train_cls = y_cls[msk]

x_test_cls = x_cls[~msk]
y_test_cls = y_cls[~msk]
x_train_cls.shape, x_test_cls.shape

In [ ]:
log_model = LogisticRegression(max_iter=1000,class_weight='balanced', random_state=42)
log_model.fit(x_train_cls, y_train_cls)
log_model

In [ ]:
y_pred_log = log_model.predict(x_test_cls)
y_pred_log

In [ ]:
y_prob_log = log_model.predict_proba(x_test_cls)[:,1]
y_prob_log

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(y_test_cls, y_pred_log))

### - Random Forest Classifier

In [ ]:
rf_cls = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_cls.fit(x_train_cls, y_train_cls)
rf_cls

In [ ]:
y_pred_rf_cls = rf_cls.predict(x_test_cls)
y_pred_rf_cls

In [ ]:
y_prob_rf = rf_cls.predict_proba(x_test_cls)[:,1]
y_prob_rf

In [ ]:
print(classification_report(y_test_cls, y_pred_rf_cls))

##  4.Regression
### - Simple Linear Regression

In [ ]:
x = df_ml[features]
y = df_ml["FutureRevenue"]
x.shape, y.shape

In [ ]:
df_ml.groupby("CutOffDate")["FutureRevenue"].agg(["count", "sum", "mean"])

In [ ]:
msk = df_ml["CutOffDate"] < '2011-10-01'
x_train = x[msk]
y_train = y[msk]

x_test = x[~msk]
y_test = y[~msk]
x_train.shape, x_test.shape

In [ ]:
scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

In [ ]:
linear_model = LinearRegression().fit(x_train_scaled, y_train)
linear_model

In [ ]:
y_pred_linear = linear_model.predict(x_test_scaled)
y_pred_linear

In [ ]:
mae = mean_absolute_error(y_test, y_pred_linear)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_linear))
r2 = r2_score(y_test, y_pred_linear)

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R2-score: {r2:.3f}")

###  - Random Forest Regression

In [ ]:
rf_model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf_model.fit(x_train, y_train)
rf_model

In [ ]:
y_pred_rf = rf_model.predict(x_test)
y_pred_rf

In [ ]:
mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)
print(f"MAE: {mae_rf:.2f}")
print(f"RMSE: {rmse_rf:.2f}")
print(f"R2 SCORE: {r2_rf:.2f}")

### - XGBoost

In [ ]:
xgb_model = XGBRegressor(n_estimators=200, random_state=42, n_jobs=-1)
xgb_model.fit(x_train, y_train)
xgb_model

In [ ]:
y_pred_xgb = xgb_model.predict(x_test)
y_pred_xgb

In [ ]:
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
r2_xgb = r2_score(y_test, y_pred_xgb)
print(f"MAE: {mae_xgb:.2f}")
print(f"RMSE: {rmse_xgb:.2f}")
print(f"R2 SCORE: {r2_xgb:.2f}")

## 5. Model Selection

### We tested 3 regression models and 2 classification models.

- Regression: Linear Regression, Random Forest, XGBoost
- Classification: Logistic Regression, Random Forest

####  - we compared the models based on their performance metrics to select the best performing model.

### Classification Model Comparison

In [ ]:
cls_results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest"
    ],
    
    "Accuracy": [
        accuracy_score(y_test_cls, y_pred_log),
        accuracy_score(y_test_cls, y_pred_rf_cls)
    ],
    
    "Precision": [
        precision_score(y_test_cls, y_pred_log),
        precision_score(y_test_cls, y_pred_rf_cls)
    ],
    
    "Recall": [
        recall_score(y_test_cls, y_pred_log),
        recall_score(y_test_cls, y_pred_rf_cls)
    ],
    
    "F1": [
        f1_score(y_test_cls, y_pred_log),
        f1_score(y_test_cls, y_pred_rf_cls)
    ],
    
    "ROC_AUC": [
        roc_auc_score(y_test_cls, y_prob_log),
        roc_auc_score(y_test_cls, y_prob_rf)
    ]
})

cls_results.sort_values("F1", ascending=False)

- Random Forest achieved higher accuracy (0.75), but performed poorly on Class 1 (Recall = 0.30, F1 = 0.40). Logistic Regression with "class_weight='balanced'" achieved lower accuracy (0.69) but significantly better Class 1 detection (Recall = 0.58, F1 = 0.51). Therefore, Logistic Regression was preferred because it provides a better balance between the two classes.

### Regression Model Comparison

In [ ]:
reg_results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Random Forest",
        "XGBoost",
    ],
    "MAE": [
        mean_absolute_error(y_test, y_pred_linear),
        mean_absolute_error(y_test, y_pred_rf),
        mean_absolute_error(y_test, y_pred_xgb),
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_test, y_pred_linear)),
        np.sqrt(mean_squared_error(y_test, y_pred_rf)),
        np.sqrt(mean_squared_error(y_test, y_pred_xgb)),
    ],
    "R2": [
        r2_score(y_test, y_pred_linear),
        r2_score(y_test, y_pred_rf),
        r2_score(y_test, y_pred_xgb),
    ]
})

reg_results.sort_values("R2", ascending=False)

- Linear Regression achieved the best overall performance, with the lowest MAE (248.12) and RMSE (926.60), and the highest R² (0.59). Random Forest performed slightly worse, while XGBoost had the weakest performance with the highest errors and lowest R² (0.31). Among the evaluated regression models, Linear Regression achieved the best performance on the holdout test set based on MAE, RMSE, and R².


### Final Results:

### The final models selected for the project were:

- Linear Regression for FutureRevenue prediction, with an R² score of approximately 0.59.
- Logistic Regression for future purchase prediction, with an ROC-AUC of approximately 0.72.

#### The models were selected after testing multiple regression and classification approaches and comparing their performance.

## 6. Feature Influence Based on Model Coefficients
### - Regression

In [ ]:
importance = pd.DataFrame({
    "Feature": x_train.columns,
    "Coefficient": linear_model.coef_
})

importance["AbsCoefficient"] = importance["Coefficient"].abs()

importance = importance.sort_values(
    "AbsCoefficient",
    ascending=False
)

importance[["Feature", "Coefficient"]]

### - Classification

In [ ]:
log_importance = pd.DataFrame({
    "Feature": x_train_cls.columns,
    "Coefficient": log_model.coef_[0]
})

log_importance["AbsCoefficient"] = log_importance["Coefficient"].abs()

log_importance = log_importance.sort_values(
    "AbsCoefficient",
    ascending=False
)

log_importance[["Feature", "Coefficient"]]

## 7. Visualization

### Distribution of Future Revenue

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df_ml["FutureRevenue"], bins=50)
plt.xlabel("Future Revenue")
plt.ylabel("Number of Customers")
plt.title("Distribution of Future Revenue")
plt.show()

### Confusion Matrix For Logistic Regression

In [ ]:
cm = confusion_matrix(y_test_cls, y_pred_log, labels=[1,0])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Purchase=1','Purchase=0'])
disp.plot(cmap='Blues')
plt.title("Confusion matrix")
plt.show()`

### ROC Curve -- Logisitic Regression

In [ ]:
from sklearn.metrics import RocCurveDisplay

RocCurveDisplay.from_predictions(
    y_test_cls,
    y_prob_log
)

plt.title("ROC Curve - Logistic Regression")
plt.show()

###  Actual vs predicted - Linear Regression

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(y_test, y_pred_linear)
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("Actual vs predicted")
plt.plot([y_test.min(), y_test.max()], [y_pred_linear.min(), y_pred_linear.max()], linestyle='--')
plt.show()

### - Linear Regression Coefficients

In [ ]:
plt.figure(figsize=(8, 5))

plt.barh(
    importance["Feature"],
    importance["AbsCoefficient"]
)

plt.xlabel("Absolute Coefficient")
plt.title("Feature Importance - Linear Regression")
plt.gca().invert_yaxis()

plt.show()

### Logistic Regression Coefficients

In [ ]:
plt.figure(figsize=(8, 5))

plt.barh(
    log_importance["Feature"],
    log_importance["AbsCoefficient"]
)

plt.xlabel("Absolute Coefficient")
plt.title("Feature Importance - Logistic Regression")
plt.gca().invert_yaxis()

plt.show()

## 8. Conclusion

In this project, we developed a machine learning workflow to predict future customer revenue and future purchase behavior using historical customer-level sales features.

To ensure a realistic evaluation and avoid data leakage, we used a time-based train-test split. Observations before "2011-10-01" were used for training, while observations from "2011-10-01" were reserved for testing. This approach better reflects a real-world scenario in which future information is not available when making predictions.

For FutureRevenue prediction, we evaluated three regression models: Linear Regression, Random Forest, and XGBoost. Linear Regression achieved the best overall performance, with an MAE of 248.12, RMSE of 926.60, and R² of 0.59. Random Forest performed slightly worse, while XGBoost achieved the weakest results on the test set. Therefore, Linear Regression was selected as the final regression model.

For future purchase prediction, we compared Logistic Regression and Random Forest. Although Random Forest achieved higher accuracy (0.75), its recall for the positive class was only 0.30. Logistic Regression achieved a lower accuracy of 0.69 but provided better detection of future purchasers, with a recall of 0.58, F1-score of 0.51, and ROC-AUC of 0.72. Therefore, Logistic Regression was selected as the final classification model because it provided a better balance between the two classes.

Feature analysis also showed that TotalRevenue had the strongest relationship with future revenue in the linear regression model, while NumberOfOrders was the most influential feature in the logistic regression model.

Overall, the project demonstrates an end-to-end machine learning workflow, including data extraction from SQL Server, feature preparation, time-based validation, model training, performance evaluation, model selection, and feature interpretation.

The results provide a useful baseline for customer-level sales prediction. However, further improvements could be achieved through additional feature engineering, hyperparameter tuning, alternative time-series validation strategies, and more advanced modeling techniques. Future work could also focus on improving the prediction of high-value customers and handling the highly skewed distribution of future revenue.